# Exercise 2 — Basic Rice Encoding (Lossless)

This notebook implements a **basic Rice encoder/decoder** for WAV PCM audio and tests:
- `Sound1.wav` and `Sound2.wav`
- `K = 2` and `K = 4`

It writes encoded files as `.ex2`, decodes them back to WAV, and checks that decoded WAV bytes exactly match the original WAV bytes.

In [7]:
from pathlib import Path
import struct
import wave
from dataclasses import dataclass

MAGIC = b"EX2R"
VERSION = 1
HEADER_STRUCT = struct.Struct("<4sBBBBIIIQ")
# fields: magic, version, k, nchannels, sampwidth, framerate, nframes, valid_bits, payload_len


def find_base_dir() -> Path:
    """Locate folder that contains Sound1.wav/Sound2.wav."""
    cwd = Path.cwd()
    candidates = [
        cwd,
        cwd / "Y3S1/CM3065 - Intelligent Signal Processing/EndTerm/Exercise2",
        Path(r"c:/Users/Enzo/Desktop/SIM-UOL/Y3S1/CM3065 - Intelligent Signal Processing/EndTerm/Exercise2"),
    ]
    for c in candidates:
        if (c / "Sound1.wav").exists():
            return c
    return cwd


BASE_DIR = find_base_dir()
print("Using BASE_DIR:", BASE_DIR)


class BitWriter:
    def __init__(self):
        self._bytes = bytearray()
        self._current = 0
        self._count = 0
        self.total_bits = 0

    def write_bit(self, bit: int):
        self._current = (self._current << 1) | (bit & 1)
        self._count += 1
        self.total_bits += 1
        if self._count == 8:
            self._bytes.append(self._current)
            self._current = 0
            self._count = 0

    def write_bits(self, value: int, nbits: int):
        for shift in range(nbits - 1, -1, -1):
            self.write_bit((value >> shift) & 1)

    def to_bytes(self) -> bytes:
        if self._count:
            self._bytes.append(self._current << (8 - self._count))
            self._current = 0
            self._count = 0
        return bytes(self._bytes)


class BitReader:
    def __init__(self, data: bytes, valid_bits: int):
        self.data = data
        self.valid_bits = valid_bits
        self.pos = 0

    def read_bit(self) -> int:
        if self.pos >= self.valid_bits:
            raise EOFError("No more bits")
        byte_idx = self.pos // 8
        bit_idx = 7 - (self.pos % 8)
        self.pos += 1
        return (self.data[byte_idx] >> bit_idx) & 1

    def read_bits(self, nbits: int) -> int:
        value = 0
        for _ in range(nbits):
            value = (value << 1) | self.read_bit()
        return value


def zigzag_encode(x: int) -> int:
    return x * 2 if x >= 0 else -2 * x - 1


def zigzag_decode(u: int) -> int:
    return u // 2 if (u % 2) == 0 else -(u // 2) - 1


def rice_encode_value(writer: BitWriter, value: int, k: int):
    q = value >> k
    r = value & ((1 << k) - 1)

    # unary(q): q zeros followed by one
    for _ in range(q):
        writer.write_bit(0)
    writer.write_bit(1)

    # remainder
    writer.write_bits(r, k)


def rice_decode_value(reader: BitReader, k: int) -> int:
    q = 0
    while reader.read_bit() == 0:
        q += 1
    r = reader.read_bits(k)
    return (q << k) | r


@dataclass
class WavData:
    nchannels: int
    sampwidth: int
    framerate: int
    nframes: int
    samples: list


def read_wav_pcm_16(path: Path) -> WavData:
    """Read 16-bit PCM WAV as signed integers."""
    with wave.open(str(path), "rb") as w:
        nchannels = w.getnchannels()
        sampwidth = w.getsampwidth()
        framerate = w.getframerate()
        nframes = w.getnframes()
        frames = w.readframes(nframes)

    if sampwidth != 2:
        raise ValueError(f"Only 16-bit PCM WAV supported in this basic notebook. Got sampwidth={sampwidth} bytes")

    total_samples = nframes * nchannels
    samples = struct.unpack("<" + "h" * total_samples, frames)
    return WavData(nchannels, sampwidth, framerate, nframes, list(samples))


def write_wav_pcm_16(path: Path, wav_data: WavData):
    if wav_data.sampwidth != 2:
        raise ValueError("Only 16-bit PCM WAV writing supported.")

    frames = struct.pack("<" + "h" * len(wav_data.samples), *wav_data.samples)
    with wave.open(str(path), "wb") as w:
        w.setnchannels(wav_data.nchannels)
        w.setsampwidth(wav_data.sampwidth)
        w.setframerate(wav_data.framerate)
        w.writeframes(frames)


def encode_wav_to_ex2(input_wav: Path, output_ex2: Path, k: int):
    wav_data = read_wav_pcm_16(input_wav)

    writer = BitWriter()

    # simple predictor: encode first sample directly, then deltas
    prev = 0
    for i, sample in enumerate(wav_data.samples):
        delta = sample - prev if i > 0 else sample
        prev = sample
        u = zigzag_encode(delta)
        rice_encode_value(writer, u, k)

    payload = writer.to_bytes()
    header = HEADER_STRUCT.pack(
        MAGIC,
        VERSION,
        k,
        wav_data.nchannels,
        wav_data.sampwidth,
        wav_data.framerate,
        wav_data.nframes,
        writer.total_bits,
        len(payload),
    )

    with open(output_ex2, "wb") as f:
        f.write(header)
        f.write(payload)


def decode_ex2_to_wav(input_ex2: Path, output_wav: Path):
    with open(input_ex2, "rb") as f:
        raw_header = f.read(HEADER_STRUCT.size)
        if len(raw_header) != HEADER_STRUCT.size:
            raise ValueError("Invalid EX2 file: header too short")

        magic, version, k, nchannels, sampwidth, framerate, nframes, valid_bits, payload_len = HEADER_STRUCT.unpack(raw_header)
        if magic != MAGIC:
            raise ValueError("Invalid EX2 file: bad magic")
        if version != VERSION:
            raise ValueError(f"Unsupported EX2 version: {version}")
        if sampwidth != 2:
            raise ValueError("Only 16-bit PCM supported by this notebook")

        payload = f.read(payload_len)
        if len(payload) != payload_len:
            raise ValueError("Invalid EX2 file: payload length mismatch")

    total_samples = nframes * nchannels
    reader = BitReader(payload, valid_bits)

    samples = []
    prev = 0
    for i in range(total_samples):
        u = rice_decode_value(reader, k)
        delta = zigzag_decode(u)
        sample = delta if i == 0 else prev + delta
        prev = sample

        # clamp just for safety against malformed bitstream
        if sample < -32768:
            sample = -32768
        elif sample > 32767:
            sample = 32767

        samples.append(sample)

    out_wav = WavData(
        nchannels=nchannels,
        sampwidth=sampwidth,
        framerate=framerate,
        nframes=nframes,
        samples=samples,
    )
    write_wav_pcm_16(output_wav, out_wav)


def files_identical(path_a: Path, path_b: Path) -> bool:
    return path_a.read_bytes() == path_b.read_bytes()


Using BASE_DIR: c:\Users\Enzo\Desktop\SIM-UOL\Y3S1\CM3065 - Intelligent Signal Processing\EndTerm\Exercise2


In [11]:
# Fix predictor to be channel-aware (important for stereo WAV interleaving)

def encode_wav_to_ex2(input_wav: Path, output_ex2: Path, k: int):
    wav_data = read_wav_pcm_16(input_wav)

    writer = BitWriter()
    prev_per_channel = [0] * wav_data.nchannels

    for i, sample in enumerate(wav_data.samples):
        ch = i % wav_data.nchannels
        delta = sample - prev_per_channel[ch]
        prev_per_channel[ch] = sample

        u = zigzag_encode(delta)
        rice_encode_value(writer, u, k)

    payload = writer.to_bytes()
    header = HEADER_STRUCT.pack(
        MAGIC,
        VERSION,
        k,
        wav_data.nchannels,
        wav_data.sampwidth,
        wav_data.framerate,
        wav_data.nframes,
        writer.total_bits,
        len(payload),
    )

    with open(output_ex2, "wb") as f:
        f.write(header)
        f.write(payload)


def decode_ex2_to_wav(input_ex2: Path, output_wav: Path):
    with open(input_ex2, "rb") as f:
        raw_header = f.read(HEADER_STRUCT.size)
        if len(raw_header) != HEADER_STRUCT.size:
            raise ValueError("Invalid EX2 file: header too short")

        magic, version, k, nchannels, sampwidth, framerate, nframes, valid_bits, payload_len = HEADER_STRUCT.unpack(raw_header)
        if magic != MAGIC:
            raise ValueError("Invalid EX2 file: bad magic")
        if version != VERSION:
            raise ValueError(f"Unsupported EX2 version: {version}")
        if sampwidth != 2:
            raise ValueError("Only 16-bit PCM supported by this notebook")

        payload = f.read(payload_len)
        if len(payload) != payload_len:
            raise ValueError("Invalid EX2 file: payload length mismatch")

    total_samples = nframes * nchannels
    reader = BitReader(payload, valid_bits)

    samples = []
    prev_per_channel = [0] * nchannels

    for i in range(total_samples):
        ch = i % nchannels
        u = rice_decode_value(reader, k)
        delta = zigzag_decode(u)
        sample = prev_per_channel[ch] + delta
        prev_per_channel[ch] = sample

        if sample < -32768:
            sample = -32768
        elif sample > 32767:
            sample = 32767

        samples.append(sample)

    out_wav = WavData(
        nchannels=nchannels,
        sampwidth=sampwidth,
        framerate=framerate,
        nframes=nframes,
        samples=samples,
    )
    write_wav_pcm_16(output_wav, out_wav)

In [12]:
# Run encode/decode for Sound1.wav and Sound2.wav with K = 4 and K = 2

sound_files = ["Sound1.wav", "Sound2.wav"]
k_values = [4, 2]
results = []

for sound in sound_files:
    input_wav = BASE_DIR / sound
    if not input_wav.exists():
        print(f"Missing file: {input_wav}")
        continue

    original_size = input_wav.stat().st_size

    row = {
        "sound": sound,
        "original_size": original_size,
    }

    for k in k_values:
        enc_path = BASE_DIR / f"{input_wav.stem}_Enc_K{k}.ex2"
        dec_path = BASE_DIR / f"{input_wav.stem}_EncDec_K{k}.wav"

        encode_wav_to_ex2(input_wav, enc_path, k)
        decode_ex2_to_wav(enc_path, dec_path)

        compressed_size = enc_path.stat().st_size
        req_equal = files_identical(input_wav, dec_path)
        compression_pct = (1 - (compressed_size / original_size)) * 100

        row[f"rice_k{k}_bytes"] = compressed_size
        row[f"compression_k{k}_pct"] = compression_pct
        row[f"lossless_k{k}"] = req_equal

    results.append(row)

results

[{'sound': 'Sound1.wav',
  'original_size': 1002088,
  'rice_k4_bytes': 857479,
  'compression_k4_pct': 14.430768555256623,
  'lossless_k4': True,
  'rice_k2_bytes': 2429933,
  'compression_k2_pct': -142.48698717078742,
  'lossless_k2': True},
 {'sound': 'Sound2.wav',
  'original_size': 1008044,
  'rice_k4_bytes': 56793316,
  'compression_k4_pct': -5534.011610604299,
  'lossless_k4': True,
  'rice_k2_bytes': 226196635,
  'compression_k2_pct': -22339.162873842808,
  'lossless_k2': True}]

In [13]:
# Pretty print table for report

if not results:
    print("No results. Check that Sound1.wav and Sound2.wav exist in BASE_DIR.")
else:
    headers = [
        "Sound",
        "Original size",
        "Rice (K=4 bits)",
        "Rice (K=2 bits)",
        "% Compression (K=4 bits)",
        "% Compression (K=2 bits)",
        "Lossless K=4",
        "Lossless K=2",
    ]

    print(" | ".join(headers))
    print("-" * 125)

    for r in results:
        print(
            f"{r['sound']} | "
            f"{r['original_size']} | "
            f"{r.get('rice_k4_bytes', 'N/A')} | "
            f"{r.get('rice_k2_bytes', 'N/A')} | "
            f"{r.get('compression_k4_pct', 0):.2f}% | "
            f"{r.get('compression_k2_pct', 0):.2f}% | "
            f"{r.get('lossless_k4', False)} | "
            f"{r.get('lossless_k2', False)}"
        )

Sound | Original size | Rice (K=4 bits) | Rice (K=2 bits) | % Compression (K=4 bits) | % Compression (K=2 bits) | Lossless K=4 | Lossless K=2
-----------------------------------------------------------------------------------------------------------------------------
Sound1.wav | 1002088 | 857479 | 2429933 | 14.43% | -142.49% | True | True
Sound2.wav | 1008044 | 56793316 | 226196635 | -5534.01% | -22339.16% | True | True
